In [9]:
from langchain_openai import ChatOpenAI
from langchain_core.messages import HumanMessage

In [10]:
llm = ChatOpenAI(model="gpt-4o-mini")
llm.invoke([HumanMessage(content="지금 몇 시야?")])    # invoke가 가능하다는 것은 llm도 Runnable하다는 뜻.


AIMessage(content='죄송하지만, 실시간으로 시간을 확인할 수는 없습니다. 하지만 시간을 확인하려면 사용 중인 기기나 시계를 확인하시면 됩니다. 도움이 필요하시면 언제든지 말씀해 주세요!', additional_kwargs={'refusal': None}, response_metadata={'token_usage': {'completion_tokens': 43, 'prompt_tokens': 13, 'total_tokens': 56, 'completion_tokens_details': {'accepted_prediction_tokens': 0, 'audio_tokens': 0, 'reasoning_tokens': 0, 'rejected_prediction_tokens': 0, 'text_tokens': None}, 'prompt_tokens_details': {'audio_tokens': 0, 'cache_write_tokens': None, 'cached_tokens': 0, 'image_tokens': None, 'text_tokens': None}}, 'model_provider': 'openai', 'model_name': 'gpt-4o-mini-2024-07-18', 'system_fingerprint': 'fp_09148a4f1e', 'id': 'chatcmpl-EYQBi6Ud3z1Q3QDwK5KsxNKkb4RxF', 'service_tier': 'default', 'finish_reason': 'stop', 'logprobs': None}, id='lc_run--01a133ec-e130-73f3-9cab-3a1d8b95cd83-0', tool_calls=[], invalid_tool_calls=[], usage_metadata={'input_tokens': 13, 'output_tokens': 43, 'total_tokens': 56, 'input_token_details': {'audio': 0, 'cache_read': 0}, 'o

In [11]:
# 현재 시간을 질문: get_current_time() function => function calling
from langchain_core.tools import tool
from datetime import datetime
import pytz

@tool   # LLM에 전달할 함수 명세서 생성
def get_current_time(timezone: str, location: str) -> str:
    """현재 시각을 반환하는 함수
    
    Args:
        timezone (str) : 타임존(예: 'Asia/Seoul'). 실제 존재해야 함.
        location (str) : 지역명. 타임존은 모든 지명에 대응되지 않으므로 이후 llm 답변 생성에 사용됨.
    """
    tz = pytz.timezone(timezone)
    now = datetime.now(tz).strftime("%Y-%m-%d %H:%M:%S")
    location_and_local_time = f"{timezone} ({location} 현재 시각 {now})"    # 반환된 값 -> LLM
    return location_and_local_time


In [12]:
# tool이 된 함수는 반드시 invoke로 실행해야 한다.
# 1. LLM에 보내지 않고 테스트 진행 => 반환 => 문자열
# 문자열  'Asia/Seoul (부산 현재 시각 2026-10-12 14:31:02)'
get_current_time.invoke({"timezone":"Asia/Seoul", "location":"부산"})
# @tool하면 get_current_time은 base tool이 되어서 바로 invoke 가능.

'Asia/Seoul (부산 현재 시각 2026-10-13 15:22:07)'

In [13]:
# 2. LLM에 보내고 받아서(tool_call) 테스트 진행. => 반환 => ToolMessage
# get_current_time.invoke(tool_call)

In [14]:
# bind_tools(tools): 모델(LLM)에게 "이런 도구(함수)들을 사용할 수 있다"고 알려주는 명세서 연결
# llm.bind_tools(함수 목록): 함수 목록: [함수, 함수] (@tool 되어있는 함수.)
# bind_tools(tools, tool_choice=None, parallel_tool_calls, strict=None)
# tools : 함수 목록
# tool_choice : 'auto', required, none, 도구 이름
# parallel_tool_call : 병렬. 한 번에 여러 도구 요청 허용 여부(True/False)
# strict : 모델이 json 스키마를 엄격하게 지키게 함(True)

# llm.bind_tools(tools).invoke()
# client -> LLM : tool 명세서 보내야 함.
# LLM -> client : 실행할 함수 목록 받아야 함.

tools=[get_current_time,]    # LLM에 보낼 함수(오브젝트) 목록. @tool된 것을 보내야 함(""감싸지X).
tool_dict = {"get_current_time":get_current_time}   # LLM이 요청한 함수명 받아서 실행할 함수와 매핑해주는 딕셔너리.
llm_with_tools = llm.bind_tools(tools)
# LLM에 보낼 준비만 한 것.

In [15]:
# 사용자의 질문과 도구를 사용해서 언어 모델(LLM) 답변 생성
from langchain_core.messages import SystemMessage

messages = [
    SystemMessage("너는 사용자의 질문에 답변하기 위해 tools를 사용할 수 있다."),
    HumanMessage("부산은 지금 몇 시야?"),
]

response=llm_with_tools.invoke(messages)
messages.append(response)
print(messages)


[SystemMessage(content='너는 사용자의 질문에 답변하기 위해 tools를 사용할 수 있다.', additional_kwargs={}, response_metadata={}), HumanMessage(content='부산은 지금 몇 시야?', additional_kwargs={}, response_metadata={}), AIMessage(content='', additional_kwargs={'refusal': None}, response_metadata={'token_usage': {'completion_tokens': 23, 'prompt_tokens': 130, 'total_tokens': 153, 'completion_tokens_details': {'accepted_prediction_tokens': 0, 'audio_tokens': 0, 'reasoning_tokens': 0, 'rejected_prediction_tokens': 0, 'text_tokens': None}, 'prompt_tokens_details': {'audio_tokens': 0, 'cache_write_tokens': None, 'cached_tokens': 0, 'image_tokens': None, 'text_tokens': None}}, 'model_provider': 'openai', 'model_name': 'gpt-4o-mini-2024-07-18', 'system_fingerprint': 'fp_17d13f9f7d', 'id': 'chatcmpl-EYQBkVSJUYgmw5cYGr8jn44ZaBqg4', 'service_tier': 'default', 'finish_reason': 'tool_calls', 'logprobs': None}, id='lc_run--01a133ec-eb77-7e83-aeca-2c1d2f142b49-0', tool_calls=[{'name': 'get_current_time', 'args': {'timezone': 'As

In [16]:
# 함수 실행 후 messages append  llm에는 아직 안 보냄.
for tool_call in response.tool_calls:
    selected_tool = tool_dict[tool_call["name"]]    # 선택된 함수 지정
    print(tool_call["args"])
    tool_message = selected_tool.invoke(tool_call)  # args 넣어서 함수 실행
    messages.append(tool_message)

messages

{'timezone': 'Asia/Seoul', 'location': '부산'}


[SystemMessage(content='너는 사용자의 질문에 답변하기 위해 tools를 사용할 수 있다.', additional_kwargs={}, response_metadata={}),
 HumanMessage(content='부산은 지금 몇 시야?', additional_kwargs={}, response_metadata={}),
 AIMessage(content='', additional_kwargs={'refusal': None}, response_metadata={'token_usage': {'completion_tokens': 23, 'prompt_tokens': 130, 'total_tokens': 153, 'completion_tokens_details': {'accepted_prediction_tokens': 0, 'audio_tokens': 0, 'reasoning_tokens': 0, 'rejected_prediction_tokens': 0, 'text_tokens': None}, 'prompt_tokens_details': {'audio_tokens': 0, 'cache_write_tokens': None, 'cached_tokens': 0, 'image_tokens': None, 'text_tokens': None}}, 'model_provider': 'openai', 'model_name': 'gpt-4o-mini-2024-07-18', 'system_fingerprint': 'fp_17d13f9f7d', 'id': 'chatcmpl-EYQBkVSJUYgmw5cYGr8jn44ZaBqg4', 'service_tier': 'default', 'finish_reason': 'tool_calls', 'logprobs': None}, id='lc_run--01a133ec-eb77-7e83-aeca-2c1d2f142b49-0', tool_calls=[{'name': 'get_current_time', 'args': {'timezone': '

In [17]:
llm_with_tools.invoke(messages)

AIMessage(content='부산 현재 시각은 2026년 10월 13일 15시 22분입니다.', additional_kwargs={'refusal': None}, response_metadata={'token_usage': {'completion_tokens': 25, 'prompt_tokens': 186, 'total_tokens': 211, 'completion_tokens_details': {'accepted_prediction_tokens': 0, 'audio_tokens': 0, 'reasoning_tokens': 0, 'rejected_prediction_tokens': 0, 'text_tokens': None}, 'prompt_tokens_details': {'audio_tokens': 0, 'cache_write_tokens': None, 'cached_tokens': 0, 'image_tokens': None, 'text_tokens': None}}, 'model_provider': 'openai', 'model_name': 'gpt-4o-mini-2024-07-18', 'system_fingerprint': 'fp_17d13f9f7d', 'id': 'chatcmpl-EYQBlxN1CR4iiuFXBPbiCGDvckhrK', 'service_tier': 'default', 'finish_reason': 'stop', 'logprobs': None}, id='lc_run--01a133ec-f014-78f2-8a17-c098ecd05d3c-0', tool_calls=[], invalid_tool_calls=[], usage_metadata={'input_tokens': 186, 'output_tokens': 25, 'total_tokens': 211, 'input_token_details': {'audio': 0, 'cache_read': 0}, 'output_token_details': {'audio': 0, 'reasoning': 0}})

In [18]:
# 주식 가격 정보
# 유효성, 형식 정의 => 클래스 생성
# def get_yf_stock_history(ticker: str, period: str): ticker: str, period: str 유효성 검증
from pydantic import BaseModel, Field
class StockHistoryInput(BaseModel): # BaseModel 상속
    ticker: str = Field(...,title="주식코드", description="주식 코드 (예: AAPL)")
    period: str = Field(..., title="기간", description="주식 데이터 조회 기간 (예: 1d, 1mo, 1y)")
    

In [19]:
# 주식 가격 가져오는 함수 선언
import yfinance as yf

@tool
# def get_yf_stock_history(ticker: str, period: str)
def get_yf_stock_history(stock_history_input: StockHistoryInput):
    """주식 종목의 가격 데이터를 조회하는 함수"""   # 함수에 대한 설명. args 줄 필요가 없어짐. pydantic 때문에.
    stock = yf.Ticker(stock_history_input.ticker)
    history = stock.history(period=stock_history_input.period)
    history_md = history.to_markdown()
    return history_md

tools = [get_current_time, get_yf_stock_history]
tool_dict = {
    "get_current_time":get_current_time,
    "get_yf_stock_history":get_yf_stock_history
}

llm_with_tools=llm.bind_tools(tools)

In [20]:
# LLM이 어떤 함수를 실행해야 하는지 선택한 함수명, args
messages.append(HumanMessage("테슬라는 한 달 전에 비해 주가가 올랐나 내렸나?"))
response = llm_with_tools.invoke(messages)
print(response)
messages.append(response)

content='' additional_kwargs={'refusal': None} response_metadata={'token_usage': {'completion_tokens': 27, 'prompt_tokens': 277, 'total_tokens': 304, 'completion_tokens_details': {'accepted_prediction_tokens': 0, 'audio_tokens': 0, 'reasoning_tokens': 0, 'rejected_prediction_tokens': 0, 'text_tokens': None}, 'prompt_tokens_details': {'audio_tokens': 0, 'cache_write_tokens': None, 'cached_tokens': 0, 'image_tokens': None, 'text_tokens': None}}, 'model_provider': 'openai', 'model_name': 'gpt-4o-mini-2024-07-18', 'system_fingerprint': 'fp_6ea874b880', 'id': 'chatcmpl-EYQBnsIJWVgLiIAwwnmZ2i9Y5MbM2', 'service_tier': 'default', 'finish_reason': 'tool_calls', 'logprobs': None} id='lc_run--01a133ec-f39a-7cd0-babc-af41c8148cc7-0' tool_calls=[{'name': 'get_yf_stock_history', 'args': {'stock_history_input': {'ticker': 'TSLA', 'period': '1mo'}}, 'id': 'call_dFt3yz8JmotZLenXuq3OtLhK', 'type': 'tool_call'}] invalid_tool_calls=[] usage_metadata={'input_tokens': 277, 'output_tokens': 27, 'total_tokens

In [21]:
# 함수 실행
# response.tool_calls[0]  # 실행함수 1개라도 원래는 for문 돌리는 게 정석. LLM이 보내는 거라 확정할 수 없음.
for tool_call in response.tool_calls:
    selected_tool = tool_dict[tool_call["name"]]    # 함수 선택
    tool_message = selected_tool.invoke(tool_call)  # invoke는 args 꺼내서 넣어줌.  함수 실행 결과. => return ToolMessage
    messages.append(tool_message)
    print(tool_message)




content='| Date                      |   Open |   High |    Low |   Close |      Volume |   Dividends |   Stock Splits |\n|:--------------------------|-------:|-------:|-------:|--------:|------------:|------------:|---------------:|\n| 2026-09-14 00:00:00-04:00 | 359.78 | 367.73 | 357.04 |  358.97 | 3.24772e+07 |           0 |              0 |\n| 2026-09-15 00:00:00-04:00 | 358.75 | 362.39 | 354.05 |  356.58 | 3.03177e+07 |           0 |              0 |\n| 2026-09-16 00:00:00-04:00 | 357.93 | 365.1  | 354.89 |  358.08 | 3.21774e+07 |           0 |              0 |\n| 2026-09-17 00:00:00-04:00 | 367.48 | 374.12 | 363.22 |  366.2  | 3.89241e+07 |           0 |              0 |\n| 2026-09-18 00:00:00-04:00 | 369    | 370.9  | 360.75 |  364.27 | 5.19223e+07 |           0 |              0 |\n| 2026-09-21 00:00:00-04:00 | 371.64 | 378.36 | 371.07 |  375.3  | 3.65996e+07 |           0 |              0 |\n| 2026-09-22 00:00:00-04:00 | 379.07 | 380.42 | 372.88 |  378.9  | 2.82299e+07 |       

In [22]:
print(llm_with_tools.invoke(messages).content)

한 달 전인 2026년 9월 14일에 테슬라(TSLA)의 종가는 358.97달러였고, 현재(2026년 10월 12일)의 종가는 378.03달러입니다.

따라서, 테슬라의 주가는 한 달 전에 비해 올랐습니다.


In [23]:
messages = [
    SystemMessage("너는 사용자의 질문에 답변하기 위해 tools를 사용할 수 있다."),
    HumanMessage("부산은 지금 몇 시야?"),
]

response=llm_with_tools.stream(messages)

is_first=True
for chunk in response:
    print("chunk type: ", type(chunk))

    if is_first:
        is_first = False
        gathered = chunk
    else:
        gathered += chunk

    print(f"content: {gathered.content}, tool_call_chunk, {gathered.tool_calls}")
messages.append(gathered)


chunk type:  <class 'langchain_core.messages.ai.AIMessageChunk'>
content: , tool_call_chunk, [{'name': 'get_current_time', 'args': {}, 'id': 'call_Q8cJiE4bDGHQWBuwZTvMvzjl', 'type': 'tool_call'}]
chunk type:  <class 'langchain_core.messages.ai.AIMessageChunk'>
content: , tool_call_chunk, [{'name': 'get_current_time', 'args': {}, 'id': 'call_Q8cJiE4bDGHQWBuwZTvMvzjl', 'type': 'tool_call'}]
chunk type:  <class 'langchain_core.messages.ai.AIMessageChunk'>
content: , tool_call_chunk, [{'name': 'get_current_time', 'args': {}, 'id': 'call_Q8cJiE4bDGHQWBuwZTvMvzjl', 'type': 'tool_call'}]
chunk type:  <class 'langchain_core.messages.ai.AIMessageChunk'>
content: , tool_call_chunk, [{'name': 'get_current_time', 'args': {'timezone': ''}, 'id': 'call_Q8cJiE4bDGHQWBuwZTvMvzjl', 'type': 'tool_call'}]
chunk type:  <class 'langchain_core.messages.ai.AIMessageChunk'>
content: , tool_call_chunk, [{'name': 'get_current_time', 'args': {'timezone': 'Asia'}, 'id': 'call_Q8cJiE4bDGHQWBuwZTvMvzjl', 'type': 't

In [24]:
# 함수 실행
for tool_call in gathered.tool_calls:
    selected_tool = tool_dict[tool_call["name"]]
    print(tool_call["args"])
    tool_msg = selected_tool.invoke(tool_call)
    messages.append(tool_msg)
messages

{'timezone': 'Asia/Seoul', 'location': '부산'}


[SystemMessage(content='너는 사용자의 질문에 답변하기 위해 tools를 사용할 수 있다.', additional_kwargs={}, response_metadata={}),
 HumanMessage(content='부산은 지금 몇 시야?', additional_kwargs={}, response_metadata={}),
 AIMessageChunk(content='', additional_kwargs={}, response_metadata={'model_provider': 'openai', 'finish_reason': 'tool_calls', 'model_name': 'gpt-4o-mini-2024-07-18', 'system_fingerprint': 'fp_6ea874b880', 'service_tier': 'default'}, id='lc_run--01a133ed-04ba-7ca1-a310-578fd5d08511', tool_calls=[{'name': 'get_current_time', 'args': {'timezone': 'Asia/Seoul', 'location': '부산'}, 'id': 'call_Q8cJiE4bDGHQWBuwZTvMvzjl', 'type': 'tool_call'}], invalid_tool_calls=[], usage_metadata={'input_tokens': 198, 'output_tokens': 23, 'total_tokens': 221, 'input_token_details': {'audio': 0, 'cache_read': 0}, 'output_token_details': {'audio': 0, 'reasoning': 0}}, tool_call_chunks=[{'name': 'get_current_time', 'args': '{"timezone":"Asia/Seoul","location":"부산"}', 'id': 'call_Q8cJiE4bDGHQWBuwZTvMvzjl', 'index': 0, 'typ

In [25]:
for c in llm_with_tools.stream(messages):
    print(c.content, end="|")

|부|산|의| 현재| 시|각|은| |202|6|년| |10|월| |13|일| |15|시| |22|분|입니다|.||||

In [26]:
# pydantic 기본 사용: 데이터 검증 -> 오류 찾기 -> 수정
# 기본 예제: User(id, name, email, age) 데이터 검증.
from pydantic import BaseModel, EmailStr, Field, ValidationError     # Field: 변수

class User(BaseModel):
    id:int                  # id는 정수를 입력받겠다. 예: "123" -> 정수 변환해줌.(pydantic이).  예2: "a" -> 에러, 기본값 없음: 필수 입력.
    name:str = "Anon"       # name은 문자열을 받겠다. 기본값으로 Anon으로 해놓았다.
    email:EmailStr          # 이메일 형식. @, 기본값 없음. => 필수입력
    age:int=Field(gt=0, lt=150) # 정수, 0 초과 150 미만, 필수 입력.

In [27]:
user = User(id="123", age=26, email="aaa@gmail.com")
print(type(user.id), user.email, user.name, user.age)

<class 'int'> aaa@gmail.com Anon 26


In [28]:
# pydantic error 발생 가능

try:
    User(id=1, email="aaa@bbb.com", age=200)
except ValidationError as e:
    print(e)

1 validation error for User
age
  Input should be less than 150 [type=less_than, input_value=200, input_type=int]
    For further information visit https://errors.pydantic.dev/2.13/v/less_than


In [29]:
# 회원가입
from pydantic import BaseModel, Field, ValidationError, EmailStr

In [30]:
class User(BaseModel):
    id:int
    name:str = "Anon"      
    email:EmailStr         
    age:int=Field(gt=0, lt=150)

In [31]:
try:
    good_user=User(
        id=123,
        email="aaa@bbb.com",
        age=25
    )
    print(f"회원가입 성공: {good_user.model_dump()}")
except ValidationError as e:
    print(e)

print("-"*50)
# 케이스 2 : 비정상 입력 => 회원가입X
try:
    bad_user = User(id=456, email="aaa", age=30)
except ValidationError as e:
    print("데이터 검증 에러 발생(오류내용)")
    print(e)


회원가입 성공: {'id': 123, 'name': 'Anon', 'email': 'aaa@bbb.com', 'age': 25}
--------------------------------------------------
데이터 검증 에러 발생(오류내용)
1 validation error for User
email
  value is not a valid email address: An email address must have an @-sign. [type=value_error, input_value='aaa', input_type=str]


In [32]:
# email 기능 확장 1
user = User(id=1, email="UserName@BBB.com", age=20)
print(user.email)

UserName@bbb.com


In [33]:
# Email 커스텀 검증: 특정 도메인(aaa.com)만 허용하는 검증
from pydantic import BaseModel, EmailStr, Field, ValidationError, field_validator

class User(BaseModel):
    id:int
    name:str="Anon"
    email:EmailStr
    age:int=Field(gt=0, lt=150)

    # 커스텀 검증하는 메소드 필요
    @field_validator("email")
    @classmethod
    def check_company_domain(cls, v:str)->str:  # v: 입력한 이메일 값
        # 특정 도메인 저장
        allowed_domain = "@aaa.com" # bbb@aaa.com

        if not v.endswith(allowed_domain):
            raise ValueError(f"이메일은 반드시 {allowed_domain} 도메인이어야 합니다.")
        return v    # 검증 성공 시 반드시 값을 반환해야 함.

In [34]:
# 케이스 1 : 오류 발생
try:
    user1=User(id=1, email="bbb@naver.com", age=30)
    print(f"이메일 인증 성공 : {user1.email}")
except ValidationError as e:
    print("이메일 검증 에러")
    print(e)

이메일 검증 에러
1 validation error for User
email
  Value error, 이메일은 반드시 @aaa.com 도메인이어야 합니다. [type=value_error, input_value='bbb@naver.com', input_type=str]
    For further information visit https://errors.pydantic.dev/2.13/v/value_error


In [35]:
# 다중 필드 검증(@model_validator)
# 패스워드, 확인 패스워드 같은지 검증
from pydantic import model_validator
class RegisterForm(BaseModel):
    password:str
    password_confirm:str

    @model_validator(mode="after")
    def check_password_match(self):
        if self.password != self.password_confirm:
            raise ValueError("비밀번호와 비밀번호 확인이 일치하지 않는다.")
        return self

In [36]:
RegisterForm(password="111",password_confirm="111")

RegisterForm(password='111', password_confirm='111')

In [37]:
# import 진행
import os
from datetime import datetime

import yfinance as yf
import pytz

from pydantic import BaseModel, Field, field_validator, ValidationError
from langchain_openai import ChatOpenAI
from langchain_core.messages import SystemMessage, HumanMessage, AIMessage,ToolMessage
from langchain_core.tools import tool

In [38]:
# 주식 매개변수(ticker, period)를 검증하는 클래스 선언
# "1d", "5d", "1mo", "3mo", "1y", "5y"
class StockHistoryInput(BaseModel):
    ticker:str = Field(..., description="주식 코드 (예: AAPL, TSLA)")
    period:str = Field(..., description="조회 기간 (예: 1d, 5d, 1mo, 1y)")

    @field_validator("period")
    @classmethod
    def check_allowed_periods(cls, value:str) -> str:
        allowed=["1d", "5d", "1mo", "3mo", "1y", "5y", "max"]

        if value not in allowed:
            raise ValueError(f"{value}는 허용되지 않는 기간입니다. 반드시 {allowed} 중 하나를 선택하세요.")
        return value

In [39]:
# tool(function) 정의

# 주식종목의 가격 조회 함수
# @tool
# def get_yf_stock_history(stock_history_input:StockHistoryInput) -> str:
@tool(args_schema=StockHistoryInput)
def get_yf_stock_history(ticker:str, period:str) -> str:
    """주식 종목의 가격 데이터를 조회하는 함수"""
    stock=yf.Ticker(ticker=ticker)
    history=stock.history(period=period)    # dataframe으로 나옴.
    return history.to_markdown()

# 현재 시각 반환하는 함수
@tool
def get_current_time(timezone:str="Asia/Seoul")->str:
    """현재 시각을 반환하는 함수"""
    return datetime.now(pytz.timezone(timezone)).strftime("%Y-%m-%d %H:%M:%S")

In [56]:
llm = ChatOpenAI(model="gpt-4o")
tools = [get_current_time, get_yf_stock_history]    # 반드시 존재해야. llm에 bind할 때
tool_dict = {   # 나중에 처리하는 클래스 생김. langgraph toolnode가 대신할 수 있음.
    "get_current_time":get_current_time, 
    "get_yf_stock_history": get_yf_stock_history
}   # llm한테 함수 이름 받아 함수 실행.
llm_with_tools = llm.bind_tools(tools)

In [72]:
# 질문 -> LLM -> tool_calls=[] 가 없으면 답변 완료
#                tool_calls=[] 가 있으면 검증, 함수 호출 => Yes(통과) : ToolMessage(함수실행결과) - invoke 후 LLM으로 감.
#                                                       => No(에러) : ToolMessage(에러문자)     - invoke 후 LLM으로 감.
# 실행 함수
def run_stock_query(query:str):     # query: 질문. "테슬라 3일 전 주가 비교해줘."
    messages = [
        SystemMessage(content="너는 주식 정보를 분석하는 비서다. " \
        "도구 호출 시 오류가 발생하면 피드백을 보고 인자를 수정하여 다시 호출하라." \
        "표의 Data 열 기준으로 답하라."),
        HumanMessage(content=query)
    ]

    print(f"질문 : {query} \n" + "-"*50)

    for i in range(3):  # 3회만 반복, range(3) = [0,1,2]
        response = llm_with_tools.invoke(messages)  # 첫 번째 질문 -> content 없고, tool_calls=[function, args=[]]
        messages.append(response)
        # tool_calls 없을 때
        if not response.tool_calls:
            print(f"최종 답변: {response.content}")
            return response.content # 함수 안에서 return은 반환 의미 & 함수 종료
        # tool_calls=[f1, f2, ...] 존재할 때
        for tool_call in response.tool_calls:
            # tool_call : name, args, id
            tool_name = tool_call["name"]   # 실행해 달라고 하는 함수명
            tool_args = tool_call["args"]   # LLM이 생성한 매개변수 값, ticker, period

            try:
                # ToolMessage create 정상 처리의 경우.
                print(f"{i+1} 회차 시도 : {tool_name}, {tool_args}")
                # period 검증. get_yf_stock_history(ticker: str, period:str)
                if tool_name == 'get_yf_stock_history': # 검증이 필요한 함수의 경우
                    # validated_args = StockHistoryInput(**tool_args) # period 검증. return value(1d,5d,1mo,...) or Error(error msg)
                    tool_message = tool_dict[tool_name].invoke(tool_call)   # ToolMessage
                else:   # 검증이 필요 없는 함수: get_current_time()
                    tool_message = tool_dict[tool_name].invoke(tool_call)
                # end ToolMessage create
                messages.append(tool_message)   # LLM에 보내기 위해 추가. 정상 함수실행결과 추가
            except (ValidationError, ValueError) as e:  # 비정상 처리의 경우
                error_str=str(e)    # error message 추출
                print(f"--> 검증 에러 발생: {error_str}")
                # 비정상처리 추가.
                messages.append(ToolMessage(
                    content=f"Valdation Error : {error_str}, 올바른 인자 값으로 다시 시도해 주세요.",  # except니까 invoke하지 않고 ToolMessage에 바로 추가.
                    tool_call_id=tool_call["id"]    # 수동으로 ToolMessage 만드는 거니까 tool_call_id 있어야 함.   
                ))
    return "재시도 횟수를 초과"


In [73]:
query="테슬라는 오늘 기준 3일 전에 비해 주가가 얼마나 올랐나 내렸나."
run_stock_query(query)

질문 : 테슬라는 오늘 기준 3일 전에 비해 주가가 얼마나 올랐나 내렸나. 
--------------------------------------------------
1 회차 시도 : get_current_time, {}
1 회차 시도 : get_yf_stock_history, {'ticker': 'TSLA', 'period': '5d'}
최종 답변: 오늘(2026년 10월 12일) 기준으로 3일 전인 2026년 10월 9일의 종가와 비교했을 때 테슬라의 주가는 다음과 같습니다:

- 2026년 10월 9일 종가: 382.70
- 2026년 10월 12일 종가: 378.03

따라서 테슬라의 주가는 약 4.67 하락하였습니다.


'오늘(2026년 10월 12일) 기준으로 3일 전인 2026년 10월 9일의 종가와 비교했을 때 테슬라의 주가는 다음과 같습니다:\n\n- 2026년 10월 9일 종가: 382.70\n- 2026년 10월 12일 종가: 378.03\n\n따라서 테슬라의 주가는 약 4.67 하락하였습니다.'